### 📖 과제 2: 감정 기반 응답 시스템

사용자의 입력을 분석하여 의도를 분류하고,
의도에 따라 다른 응답을 생성하는 조건부 챗봇을 구현하세요.

**요구사항**:

1. State에 user_text, intent, response 필드 정의
2. analyze_intent 노드: 입력을 다음 중 하나로 분류
    - question(인사 혹은 일반 질문) / complaint(불만/클레임) / request(요청)
3. route_by_intent 함수: intent 기반 라우팅
4. 3개의 응답 노드 구현
    - respond_question
    - respond_complaint
    - respond_request
5. 조건부 엣지(add_conditional_edges)로 연결

**예시**:
- 입력: "이거 어떻게 사용하나요?"
- 의도: question
- 응답: 설명형 답변

In [ ]:
# CODE HERE

<details>

<summary>모범 답안 펼쳐보기</summary>

```python
from typing import Literal
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage

# LLM 초기화
llm = init_chat_model("gpt-5.4-mini")

# State 정의
class State(TypedDict):
    user_text: str
    intent: str
    response: str

# 의도 분석 노드
def analyze_intent(state: State):
    print(f"의도 분석 중: '{state['user_text']}'")

    messages = [
        SystemMessage(content="""사용자 입력을 분석하여 다음 중 하나로 분류하세요:
- question: 인사, 일반 질문, 정보 요청
- complaint: 불만, 클레임, 부정적 피드백
- request: 특정 작업 요청, 요구사항

'question', 'complaint', 'request' 중 하나의 단어만 반환하세요."""),
        HumanMessage(content=state["user_text"])
    ]

    result = llm.invoke(messages)
    intent = result.content.strip().lower()

    print(f"의도 분석 결과: {intent}")
    return {"intent": intent}

# 의도에 따라 경로 결정
def route_by_intent(state: State) -> Literal["question", "complaint", "request"]:
    """의도에 따라 다음 노드를 결정"""
    intent = state["intent"]

    if "question" in intent:
        return "question"
    elif "complaint" in intent:
        return "complaint"
    else:
        return "request"

# 질문 응답 노드
def respond_question(state: State):
    print("질문 응답 생성 중...")

    messages = [
        SystemMessage(content="""
                      당신은 고객의 문의 내용을 처리하는 챗봇이며, 사용자로부터 일반 질문 혹은 정보 요청이 들어왔습니다.
                      사용자의 질문에 명확하고 이해하기 쉽게 답변하세요."""),
        HumanMessage(content=state["user_text"])
    ]

    result = llm.invoke(messages)
    response = result.content

    return {"response": response}

# 불만 응답 노드
def respond_complaint(state: State):
    print("불만 응답 생성 중...")

    messages = [
        SystemMessage(content="""
                      당신은 고객의 문의 내용을 처리하는 챗봇이며, 사용자로부터 클레임이 들어왔습니다.
                      사용자의 불만을 경청하고, 사과하며, 해결 방안을 제시하세요."""),
        HumanMessage(content=state["user_text"])
    ]

    result = llm.invoke(messages)
    response = result.content

    return {"response": response}

# 요청 응답 노드
def respond_request(state: State):
    print("요청 응답 생성 중...")

    messages = [
        SystemMessage(content="""
                      당신은 고객의 문의 내용을 처리하는 챗봇이며, 사용자로부터 특정 작업 요청이 들어왔습니다.
                      사용자의 요청을 확인하고, 처리 방법과 다음 단계를 안내해주세요."""),
        HumanMessage(content=state["user_text"])
    ]

    result = llm.invoke(messages)
    response = result.content

    return {"response": response}

# 그래프 구성
graph_builder = StateGraph(State)

# 노드 추가
graph_builder.add_node("analyze_intent", analyze_intent)
graph_builder.add_node("question", respond_question)
graph_builder.add_node("complaint", respond_complaint)
graph_builder.add_node("request", respond_request)

# 엣지 연결
graph_builder.add_edge(START, "analyze_intent")

# 조건부 엣지
graph_builder.add_conditional_edges(
    "analyze_intent",
    route_by_intent,
    {
        "question": "question",
        "complaint": "complaint",
        "request": "request",
    }
)

# 모든 응답 노드를 END로 연결
graph_builder.add_edge("question", END)
graph_builder.add_edge("complaint", END)
graph_builder.add_edge("request", END)

# 그래프 컴파일
graph = graph_builder.compile()

# 시각화
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print(e)

# 다양한 텍스트로 테스트
test_texts = [
    "이거 어떻게 사용하나요?",
    "이 제품 배송이 너무 늦어요. 정말 화가 나네요.",
    "환불 처리 좀 해주세요. 제품이 불량입니다."
]

for text in test_texts:
    print("\n" + "="*70)
    result = graph.invoke({
        "user_text": text,
        "intent": "",
        "response": ""
    })

    print(f"입력: {result['user_text']}")
    print(f"의도: {result['intent']}")
    print(f"응답: {result['response']}")
```
</details>